In [1]:
import os
import re

# concat_xdatcar.py

BASE_DIR = "/home/nico/MD_trajectories"
OUT_DIR = os.getcwd()  # project directory where XDATCAR_... will be written

def extract_index(name):
    m = re.findall(r'(\d+)', name)
    return int(m[-1]) if m else -1

def temp_from_folder(name):
    m = re.search(r'(\d+)', name)
    return m.group(1) if m else name

for entry in sorted(os.scandir(BASE_DIR), key=lambda e: e.name):
    if not entry.is_dir():
        continue
    temp = temp_from_folder(entry.name)
    files = [f for f in os.listdir(entry.path) if os.path.isfile(os.path.join(entry.path, f)) and f.lower().startswith("xdatcar")]
    if not files:
        continue
    files_sorted = sorted(files, key=extract_index)
    out_path = os.path.join(OUT_DIR, f"XDATCAR_{temp}")
    with open(out_path, "w", encoding="utf-8") as outf:
        for i, fname in enumerate(files_sorted):
            fp = os.path.join(entry.path, fname)
            with open(fp, "r", encoding="utf-8", errors="replace") as inf:
                lines = inf.readlines()
            if i == 0:
                outf.writelines(lines)
            else:
                # Skip header: find first "Direct configuration" (case-insensitive) and append from there
                idx = None
                for j, line in enumerate(lines):
                    if re.match(r'^\s*Direct configuration', line, re.I):
                        idx = j
                        break
                if idx is None:
                    # fallback: look for a line that starts with "Direct" or contains "Direct configuration="
                    for j, line in enumerate(lines):
                        if re.match(r'^\s*Direct\b', line, re.I) or "Direct configuration=" in line:
                            idx = j
                            break
                if idx is None:
                    outf.writelines(lines)
                else:
                    outf.writelines(lines[idx:])
    print(f"Wrote {out_path}")

Wrote /home/nico/bachelor-thesis/XDATCAR_300
Wrote /home/nico/bachelor-thesis/XDATCAR_520
Wrote /home/nico/bachelor-thesis/XDATCAR_650


In [2]:
import glob
import os
import numpy as np
import MDAnalysis as mda
from MDAnalysis.analysis import rms
from pymbar import timeseries as ts

# analyze_equilibration.py

try:
    HAVE_PYMBAR = True
except Exception:
    HAVE_PYMBAR = False

# OUT_DIR is available in the notebook; fallback if not present
OUT_DIR = globals().get("OUT_DIR", os.getcwd())

def detect_equilibration_simple(x, tol_factor=0.1, min_fraction=0.1):
    # fallback equil detection: find first frame after which
    # a windowed mean stays within tol of final mean
    final_mean = x.mean()
    tol = x.std() * tol_factor
    window = max(1, int(len(x) * min_fraction))
    for i in range(len(x) - window):
        if abs(x[i:i+window].mean() - final_mean) < tol:
            return i
    return 0

results = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, "XDATCAR_*"))):
    try:
        u = mda.Universe(path)  # XDATCAR reader is auto-detected
    except Exception as e:
        print(f"SKIP {path}: cannot read ({e})")
        continue

    # compute RMSD vs frame 0 for all atoms
    R = rms.RMSD(u, u, select="all", ref_frame=0).run()
    rmsd = R.rmsd[:, 2]  # column 2 is RMSD in Å

    if len(rmsd) == 0:
        print(f"NO FRAMES {path}")
        continue

    if HAVE_PYMBAR:
        eq, g, Neff = ts.detectEquilibration(rmsd)
        eq_frame = int(eq)
        method = "pymbar.detectEquilibration"
    else:
        eq_frame = detect_equilibration_simple(rmsd)
        g = None
        Neff = None
        method = "simple_window"

    results.append((path, len(rmsd), eq_frame, method, g, Neff, rmsd.mean(), rmsd.std()))
    print(f"{os.path.basename(path)}: frames={len(rmsd)} eq_frame={eq_frame} method={method}")

# results list contains tuples:
# (path, n_frames, equil_frame, method, g, Neff, mean_rmsd, std_rmsd)

/home/nico/miniforge3/envs/NPW/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX

SKIP /home/nico/bachelor-thesis/XDATCAR_300: cannot read ('' isn't a valid topology format, nor a coordinate format
   from which a topology can be minimally inferred.
   You can use 'Universe(topology, ..., topology_format=FORMAT)'
   to explicitly specify the format and
   override automatic detection. Known FORMATs are:
   dict_keys(['PSF', 'TOP', 'PRMTOP', 'PARM7', 'PDB', 'ENT', 'XPDB', 'PQR', 'GRO', 'CRD', 'PDBQT', 'DMS', 'TPR', 'MOL2', 'DATA', 'LAMMPSDUMP', 'XYZ', 'TXYZ', 'ARC', 'GMS', 'CONFIG', 'HISTORY', 'XML', 'MMTF', 'GSD', 'MINIMAL', 'ITP', 'IN', 'FHIAIMS', 'PARMED', 'RDKIT', 'OPENMMTOPOLOGY', 'OPENMMAPP'])
   See https://docs.mdanalysis.org/documentation_pages/topology/init.html#supported-topology-formats
   For missing formats, raise an issue at 
   https://github.com/MDAnalysis/mdanalysis/issues)
SKIP /home/nico/bachelor-thesis/XDATCAR_520: cannot read ('' isn't a valid topology format, nor a coordinate format
   from which a topology can be minimally inferred.
   You can